In [ ]:
import os

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml
from mpl_toolkits.axes_grid1 import make_axes_locatable

with open("panel_sizes_cm.yaml", "r") as file:
    panel_sizes = yaml.safe_load(file)

In [ ]:
metric_header = {
    "neg_log_marginal_NLE": "log w",
}

In [ ]:
# Load the results of the weight evaluation
df = pd.read_csv("../compute_metrics_with_gt_likelihood.csv")

# Set the evaluated metric
metric_name = "neg_log_marginal_NLE"

# Temperature used in the computation of the resampling weights
T = 1.0

# Get the random seeds for which the evaluation was conducted
seeds = df["seed"].unique()
print(f"Found {len(seeds)} unique random seeds.")

n_candidates = len(df["idx_ground_truth_model"].unique())
print(f"Found {n_candidates} unique target models.")

# For each evaluated seeds, collect the pairwise log-resampling weights
plots = []

for seed in seeds:
    # Initialize empty panel
    plot_seed = torch.zeros([n_candidates, n_candidates])

    for idx_gt in range(n_candidates):
        # Select the entries that have the current target model
        df_idx = df[df["idx_ground_truth_model"] == idx_gt]

        # Select the entries that also have the correct random seed
        df_idx = df_idx[df_idx["seed"] == seed]

        # Extract the raw metric values
        metric_vals = -torch.from_numpy(df_idx[metric_name].values)

        # Compute the log boltzmann weight
        log_w = metric_vals * T
        log_w = log_w - torch.logsumexp(log_w, 0)

        assert np.isclose(log_w.exp().sum(), 1.0)

        # For each combination of seed and target model, there should be
        # n_components entries
        assert n_candidates == len(log_w)

        # Each target model is one row in the panel
        plot_seed[idx_gt] = log_w

    # Collect the results
    plots.append(plot_seed)


# Average over all seeds
sum_plot = torch.zeros([n_candidates, n_candidates])

for plot in plots:
    sum_plot += plot
sum_plot /= len(plots)

cmap = "Blues"

f = "../../"
style_file = os.path.join(f, ".matplotlibrc")

with mpl.rc_context(fname=style_file):
    fig, ax = plt.subplots(
        1,
        1,
        figsize=(
            panel_sizes["panel_resampling_weights"]["width_cm"] / 2.54,
            panel_sizes["panel_resampling_weights"]["height_cm"] / 2.54,
        ),
    )

    im = ax.imshow(sum_plot, cmap=cmap)

    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size="5%", pad=0.05)

    ax.set_xlabel("candidate model")
    ax.set_ylabel("target model")
    ax.set_title(metric_header[metric_name])
    cbar = plt.colorbar(mappable=im, shrink=0.75, cax=cax)
    cbar.ax.yaxis.set_major_formatter(lambda x, _: f"{x:.0e}")

    fig.tight_layout()

    fig.savefig(
        "../panels/resampling_weights_gt_likelihood.svg",
        bbox_inches="tight",
        format="svg",
        transparent=True,
    )